### IMDB report tables generator


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

all_results = pd.read_csv("project_outputs/tables/all_model_results.csv")
ablation_results = pd.read_csv("project_outputs/tables/ablation_results.csv")

print("Loaded")
print()
print("all_model_results shape:", all_results.shape)
print("ablation_results shape:", ablation_results.shape)


Loaded

all_model_results shape: (104, 11)
ablation_results shape: (12, 9)


### 1 Dataset length statistics

load the IMDB dataset directly


In [10]:

texts = None

try:
    from datasets import load_dataset
    ds = load_dataset("imdb")
    texts = list(ds["train"]["text"]) + list(ds["test"]["text"])
    print(f"Loaded IMDB directly from datasets: {len(texts):,} reviews")
except Exception as e:
    print("Direct dataset load failed.")
    print("Replace this block with your runner.py dataset loading code.")
    print("Error:", e)



c:\Users\psadi\anaconda3\envs\ds6050\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded IMDB directly from datasets: 50,000 reviews


In [ ]:
if texts is None:
    raise ValueError("Set `texts` first using your runner.py loading code or the datasets package.")

word_lengths = [len(str(t).split()) for t in texts]

# Token lengths using RoBERTa tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
token_lengths = [len(tokenizer.encode(str(t), truncation=False)) for t in texts]

def summarize(lengths, name):
    return {
        "Length type": name,
        "Min": int(np.min(lengths)),
        "Median": int(np.median(lengths)),
        "Mean": round(float(np.mean(lengths)), 2),
        "Max": int(np.max(lengths)),
    }

length_stats = pd.DataFrame([
    summarize(word_lengths, "Words"),
    summarize(token_lengths, "Tokens"),
])

length_stats


Token indices sequence length is longer than the specified maximum sequence length for this model (680 > 512). Running this sequence through the model will result in indexing errors


,Length type,Min,Median,Mean,Max
0,Words,4,173,231.16,2470
1,Tokens,10,222,298.25,3099


In [12]:
def length_bin(token_len):
    if token_len < 128:
        return "short"
    elif token_len <= 512:
        return "medium"
    else:
        return "long"

bin_counts = (
    pd.Series(token_lengths, name="token_length")
      .map(length_bin)
      .value_counts()
      .reindex(["short", "medium", "long"])
      .rename_axis("Length bin")
      .reset_index(name="Review count")
)

bin_counts["Percent"] = (100 * bin_counts["Review count"] / bin_counts["Review count"].sum()).round(2)
bin_counts


,Length bin,Review count,Percent
0,short,5752,11.50
1,medium,37643,75.29
2,long,6605,13.21


### 2 Main model comparison table

- Logistic Regression
- BiLSTM_T128_head+tail
- RoBERTa_T512_head-only


In [13]:
primary_models = [
    "Logistic Regression",
    "BiLSTM_T128_head+tail",
    "RoBERTa_T512_head-only",
]

overall_test_table = (
    all_results[
        (all_results["split"] == "test") &
        (all_results["metric_scope"] == "overall") &
        (all_results["model"].isin(primary_models))
    ][["model", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values("model")
)

overall_test_table = overall_test_table.rename(columns={
    "model": "Model",
    "accuracy": "Accuracy",
    "f1": "F1",
    "auc": "AUC",
    "training_time_seconds": "Train time (s)",
    "inference_time_seconds": "Inference time (s)",
})

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    overall_test_table[col] = overall_test_table[col].round(4)

overall_test_table


,Model,Accuracy,F1,AUC,Train time (s),Inference time (s)
20,BiLSTM_T128_head+tail,0.7961,0.8000,0.8782,4.0180,2.3648
4,Logistic Regression,0.8918,0.8924,0.9579,6.7234,4.2722
92,RoBERTa_T512_head-only,0.9467,0.9469,0.9869,307.4250,134.6390


### 3 F1 by review length group



In [14]:
f1_by_length_table = (
    all_results[
        (all_results["split"] == "test") &
        (all_results["metric_scope"] == "by_length") &
        (all_results["model"].isin(primary_models))
    ][["model", "length_bin", "f1"]]
    .copy()
    .pivot(index="model", columns="length_bin", values="f1")
    .reindex(primary_models)
    [["short", "medium", "long"]]
    .reset_index()
)

f1_by_length_table.columns = ["Model", "Short F1", "Medium F1", "Long F1"]
for col in ["Short F1", "Medium F1", "Long F1"]:
    f1_by_length_table[col] = f1_by_length_table[col].round(4)

f1_by_length_table


,Model,Short F1,Medium F1,Long F1
0,Logistic Regression,0.9047,0.8875,0.8942
1,BiLSTM_T128_head+tail,0.8257,0.7944,0.7655
2,RoBERTa_T512_head-only,0.9495,0.9487,0.9250


### 4 Ablation tables



In [15]:
bilstm_ablation = (
    ablation_results[ablation_results["model_family"] == "BiLSTM"]
    [["max_length", "truncation_strategy", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values(["max_length", "truncation_strategy"])
    .rename(columns={
        "max_length": "Max length",
        "truncation_strategy": "Truncation",
        "accuracy": "Accuracy",
        "f1": "F1",
        "auc": "AUC",
        "training_time_seconds": "Train time (s)",
        "inference_time_seconds": "Inference time (s)",
    })
)

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    bilstm_ablation[col] = bilstm_ablation[col].round(4)

bilstm_ablation


,Max length,Truncation,Accuracy,F1,AUC,Train time (s),Inference time (s)
1,128,head+tail,0.7961,0.8000,0.8782,4.0180,2.3648
0,128,head-only,0.7300,0.7498,0.8148,4.0393,2.2052
3,256,head+tail,0.7520,0.7580,0.8222,4.4970,2.8313
2,256,head-only,0.6554,0.6695,0.7179,4.4793,2.8173
5,512,head+tail,0.7477,0.7495,0.8188,5.1950,3.7334
4,512,head-only,0.7137,0.6947,0.7773,5.1555,3.6367


In [16]:
roberta_ablation = (
    ablation_results[ablation_results["model_family"] == "RoBERTa"]
    [["max_length", "truncation_strategy", "accuracy", "f1", "auc", "training_time_seconds", "inference_time_seconds"]]
    .copy()
    .sort_values(["max_length", "truncation_strategy"])
    .rename(columns={
        "max_length": "Max length",
        "truncation_strategy": "Truncation",
        "accuracy": "Accuracy",
        "f1": "F1",
        "auc": "AUC",
        "training_time_seconds": "Train time (s)",
        "inference_time_seconds": "Inference time (s)",
    })
)

for col in ["Accuracy", "F1", "AUC", "Train time (s)", "Inference time (s)"]:
    roberta_ablation[col] = roberta_ablation[col].round(4)

roberta_ablation


,Max length,Truncation,Accuracy,F1,AUC,Train time (s),Inference time (s)
7,128,head+tail,0.9234,0.9222,0.9775,101.2938,37.5598
6,128,head-only,0.8983,0.8960,0.9660,100.3546,40.2628
9,256,head+tail,0.9231,0.9274,0.9855,163.2012,65.1802
8,256,head-only,0.9298,0.9299,0.9799,165.8410,66.7980
11,512,head+tail,0.9467,0.9456,0.9879,316.4033,142.1488
10,512,head-only,0.9467,0.9469,0.9869,307.4250,134.6390


### 5 Compact efficiency table


In [17]:
efficiency_table = overall_test_table[["Model", "Train time (s)", "Inference time (s)"]].copy()
efficiency_table


,Model,Train time (s),Inference time (s)
20,BiLSTM_T128_head+tail,4.0180,2.3648
4,Logistic Regression,6.7234,4.2722
92,RoBERTa_T512_head-only,307.4250,134.6390


### 6 Optional: export all tables to CSV and LaTeX



In [19]:
out_dir = Path("project_outputs/report_tables")
out_dir.mkdir(exist_ok=True)

tables = {
    "length_stats": length_stats,
    "bin_counts": bin_counts,
    "overall_test_table": overall_test_table,
    "f1_by_length_table": f1_by_length_table,
    "bilstm_ablation_table": bilstm_ablation,
    "roberta_ablation_table": roberta_ablation,
    "efficiency_table": efficiency_table,
}

for name, df in tables.items():
    csv_path = out_dir / f"{name}.csv"
    tex_path = out_dir / f"{name}.tex"
    
    df.to_csv(csv_path, index=False)
    
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, escape=False))
    
    print(f"Saved {csv_path.resolve()}")
    print(f"Saved {tex_path.resolve()}")


Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\length_stats.csv
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\length_stats.tex
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\bin_counts.csv
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\bin_counts.tex
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\overall_test_table.csv
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\overall_test_table.tex
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\f1_by_length_table.csv
Saved C:\Users\psadi\OneDrive\Desktop\MSDS\DS 6050\DS6050-imdb_sentiment\project_outputs\report_tables\f1_by_length_table.tex
Saved C:\Users\psadi